<a href="https://colab.research.google.com/github/ibtisamelahi07/generativeai/blob/main/Copy_of_RAG_Unsloth_Dynamic4bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Task 3: RAG Pipeline with Unsloth's Dynamic 4-bit Quantization

**Pipeline Overview:**
1. Load Unsloth dynamic 4-bit quantized LLM (VRAM-efficient)
2. Index domain-specific documents using sentence embeddings
3. Build FAISS retrieval index for semantic search
4. Implement a RAG pipeline that retrieves relevant chunks and generates grounded responses

**Runtime:** GPU (T4 recommended — 16 GB VRAM)  
**Model:** `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`  
**Domain:** Operations Research & Industrial Engineering (customizable)

---
## 🔧 Section 1: Install Dependencies

In [ ]:
%%capture
# Core: Unsloth for dynamic 4-bit quantization
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

# RAG stack
!pip install sentence-transformers faiss-cpu

# Utilities
!pip install numpy pandas

print("All dependencies installed.")

---
## 🖥️ Section 2: GPU & VRAM Verification

In [ ]:
import torch

def print_vram_status(label=""):
    """Utility to print current VRAM usage at any checkpoint."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        free      = total - reserved
        print(f"{'[' + label + '] ' if label else ''}VRAM — "
              f"Allocated: {allocated:.2f} GB | "
              f"Reserved: {reserved:.2f} GB | "
              f"Free: {free:.2f} GB / {total:.2f} GB total")
    else:
        print("⚠️  No GPU detected — switch to GPU runtime (Runtime > Change runtime type > T4 GPU).")

if torch.cuda.is_available():
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print_vram_status("Baseline")
else:
    raise RuntimeError("GPU required. Enable GPU runtime before proceeding.")

✅ GPU detected: Tesla T4
[Baseline] VRAM — Allocated: 0.00 GB | Reserved: 0.00 GB | Free: 15.64 GB / 15.64 GB total


---
## 🤖 Section 3: Load Unsloth Dynamic 4-bit Quantized Model



In [ ]:
from unsloth import FastLanguageModel

# ── Configuration ────────────────────────────────────────────────
MODEL_NAME     = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # Dynamic bnb-4bit variant
MAX_SEQ_LENGTH = 2048   # Covers long retrieved contexts
DTYPE          = None   # None → auto-detect (float16 on T4, bfloat16 on A100)
LOAD_IN_4BIT   = True   # Enable dynamic 4-bit quantization
# ─────────────────────────────────────────────────────────────────

print(f"Loading model: {MODEL_NAME}")
print("Dynamic 4-bit quantization: ENABLED")
print("This selectively preserves precision for critical parameters...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

# Switch to optimised inference mode (disables gradient tracking, enables flash-attn)
FastLanguageModel.for_inference(model)

print("\n✅ Model loaded and set to inference mode.")
print_vram_status("After model load")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: unsloth/Llama-3.2-3B-Instruct-bnb-4bit
Dynamic 4-bit quantization: ENABLED
This selectively preserves precision for critical parameters...

==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.



✅ Model loaded and set to inference mode.
[After model load] VRAM — Allocated: 2.29 GB | Reserved: 2.33 GB | Free: 13.31 GB / 15.64 GB total


---
## 📚 Section 4: Domain-Specific Document Corpus

The corpus below covers **Operations Research & Industrial Engineering** concepts.  
> **Customisation tip:** Replace `DOCUMENTS` with any list of strings from your own domain (PDFs, textbooks, notes, etc.).

In [ ]:
# ── Domain corpus: Operations Research & Industrial Engineering ──
# Each string is one logical document / section.
# Swap or extend this list to change the knowledge domain.

DOCUMENTS = [
    # ── Linear Programming ──────────────────────────────────────
    """Linear Programming (LP) is a mathematical optimisation technique used to find
the best outcome given a set of linear constraints. The objective function
represents the quantity to be maximised or minimised (e.g., profit, cost).
Decision variables represent controllable inputs. Constraints are linear
inequalities or equalities that bound feasible solutions. The simplex method,
developed by George Dantzig in 1947, iterates over corner points (vertices)
of the feasible polytope to find the optimum. Interior-point methods such as
Karmarkar's algorithm provide polynomial-time alternatives.""",

    # ── Integer Programming ─────────────────────────────────────
    """Integer Programming (IP) extends LP by requiring some or all decision
variables to take integer values. When all variables are binary (0 or 1),
the problem is a Binary Integer Program (BIP), useful for yes/no decisions
such as facility location and project selection. Mixed-Integer Programs (MIPs)
combine continuous and integer variables. Branch-and-bound and cutting-plane
methods are standard solution algorithms. Commercial solvers like Gurobi and
CPLEX handle large-scale MIPs efficiently.""",

    # ── Queueing Theory ─────────────────────────────────────────
    """Queueing theory analyses waiting lines (queues) in systems such as call
centres, hospitals, and ticket platforms. The M/M/1 model assumes Poisson
arrivals, exponential service times, and a single server. The M/M/c model
generalises this to c parallel servers, commonly used for multi-agent customer
service systems. Key performance metrics include: average queue length (Lq),
average waiting time (Wq), server utilisation (ρ = λ/cμ), and probability of
an idle system (P0). Erlang-C formula gives the probability a customer must
wait in an M/M/c queue.""",

    # ── Goal Programming ────────────────────────────────────────
    """Goal Programming (GP) is a multi-objective optimisation technique that
minimises deviations from target (goal) levels for each objective. Deviation
variables d+ (overachievement) and d- (underachievement) capture the gap
between achieved and desired levels. In weighted GP, the objective function
is a weighted sum of deviations. In lexicographic GP, objectives are ranked
by priority and optimised sequentially. GP is widely applied in resource
allocation, personnel scheduling, and financial planning.""",

    # ── Dynamic Programming ─────────────────────────────────────
    """Dynamic Programming (DP) solves complex problems by breaking them into
overlapping subproblems and storing solutions to avoid redundant computation
(memoisation). Bellman's principle of optimality states that an optimal policy
has the property that, whatever the initial state and decision, the remaining
decisions must constitute an optimal policy for the subproblem. DP is applied
to inventory control, capital budgeting, network shortest-path problems, and
stochastic sequential decision processes (Markov Decision Processes).""",

    # ── Network Models ──────────────────────────────────────────
    """Network optimisation models represent problems as graphs of nodes and arcs.
The shortest-path problem (solved by Dijkstra's or Floyd-Warshall algorithms)
finds minimum-cost routes. The minimum spanning tree problem (Kruskal's,
Prim's) connects all nodes at minimum total arc cost. The maximum-flow problem
determines the greatest flow through a network from source to sink. The
transportation problem minimises the cost of shipping goods from origins to
destinations and is a special-structure LP solvable efficiently by the
transportation simplex method.""",

    # ── Simulation ──────────────────────────────────────────────
    """Monte Carlo simulation uses random sampling to estimate mathematical
quantities and model stochastic systems. In Industrial Engineering, discrete-
event simulation (DES) models queues, manufacturing lines, and logistics
networks over time. Tools include Arena, SimPy (Python), and AnyLogic.
Variance-reduction techniques such as antithetic variates and control variates
improve estimation efficiency. A sufficient number of replications is required
to obtain statistically valid confidence intervals on performance metrics.""",

    # ── Lean & Six Sigma ────────────────────────────────────────
    """Lean manufacturing is derived from the Toyota Production System and focuses
on eliminating waste (muda): overproduction, waiting, transportation, over-
processing, inventory, motion, and defects. Value Stream Mapping (VSM)
visualises material and information flows. Six Sigma uses DMAIC (Define,
Measure, Analyse, Improve, Control) to reduce process variation. Combined as
Lean Six Sigma, the methodology improves efficiency and quality simultaneously.
Statistical process control (SPC) charts monitor process stability over time.""",

    # ── Inventory Management ────────────────────────────────────
    """Inventory management balances holding costs against ordering (setup) costs
and stockout risks. The Economic Order Quantity (EOQ) model minimises total
inventory cost for deterministic, constant demand. The Reorder Point (ROP)
triggers a replenishment order when stock falls to a safety-stock level.
Probabilistic models (newsvendor, (s,S) policies) handle stochastic demand.
ABC analysis classifies items by value to concentrate control effort. Just-In-
Time (JIT) inventory reduces waste by synchronising supply with production.""",

    # ── Probability Models ──────────────────────────────────────
    """Probability models quantify uncertainty in engineering and management
systems. Common distributions include: Poisson (arrival counts), Exponential
(service/inter-arrival times), Normal (measurement errors), Binomial (success/
failure counts), and Uniform (equally likely outcomes). Markov chains model
systems that transition between states with fixed probabilities, regardless of
history (memoryless property). Steady-state probabilities describe long-run
behaviour. Bayesian inference updates prior beliefs using observed data via
Bayes' theorem: P(H|E) = P(E|H)·P(H) / P(E).""",
]

print(f"✅ Corpus loaded: {len(DOCUMENTS)} domain documents.")
for i, doc in enumerate(DOCUMENTS):
    preview = doc.strip().replace('\n', ' ')[:80]
    print(f"  Doc {i+1:02d}: {preview}...")

✅ Corpus loaded: 10 domain documents.
  Doc 01: Linear Programming (LP) is a mathematical optimisation technique used to find th...
  Doc 02: Integer Programming (IP) extends LP by requiring some or all decision variables ...
  Doc 03: Queueing theory analyses waiting lines (queues) in systems such as call centres,...
  Doc 04: Goal Programming (GP) is a multi-objective optimisation technique that minimises...
  Doc 05: Dynamic Programming (DP) solves complex problems by breaking them into overlappi...
  Doc 06: Network optimisation models represent problems as graphs of nodes and arcs. The ...
  Doc 07: Monte Carlo simulation uses random sampling to estimate mathematical quantities ...
  Doc 08: Lean manufacturing is derived from the Toyota Production System and focuses on e...
  Doc 09: Inventory management balances holding costs against ordering (setup) costs and s...
  Doc 10: Probability models quantify uncertainty in engineering and management systems. C...


---
##  Section 5: Text Chunking

Long documents are split into overlapping chunks so that retrieval can pinpoint the most relevant passage without truncation.

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    """
    Split text into word-level chunks with sliding overlap.

    Args:
        text       : Input document string.
        chunk_size : Number of words per chunk.
        overlap    : Number of overlapping words between consecutive chunks.

    Returns:
        List of chunk strings.
    """
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap  # slide forward by (chunk_size - overlap)
    return chunks


# ── Chunk all documents ─────────────────────────────────────────
CHUNK_SIZE = 150   # words per chunk
OVERLAP    = 30    # overlapping words for context continuity

all_chunks = []        # flat list of chunk strings
chunk_metadata = []    # parallel list: {doc_id, doc_preview}

for doc_id, doc in enumerate(DOCUMENTS):
    chunks = chunk_text(doc, CHUNK_SIZE, OVERLAP)
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_metadata.append({
            "doc_id"      : doc_id,
            "doc_preview" : doc.strip().split('\n')[0][:50]  # first line as label
        })

print(f"✅ Chunking complete.")
print(f"   Total chunks : {len(all_chunks)}")
print(f"   Chunk size   : ~{CHUNK_SIZE} words with {OVERLAP}-word overlap")
print(f"\nSample chunk:\n{all_chunks[0]}")

✅ Chunking complete.
   Total chunks : 10
   Chunk size   : ~150 words with 30-word overlap

Sample chunk:
Linear Programming (LP) is a mathematical optimisation technique used to find the best outcome given a set of linear constraints. The objective function represents the quantity to be maximised or minimised (e.g., profit, cost). Decision variables represent controllable inputs. Constraints are linear inequalities or equalities that bound feasible solutions. The simplex method, developed by George Dantzig in 1947, iterates over corner points (vertices) of the feasible polytope to find the optimum. Interior-point methods such as Karmarkar's algorithm provide polynomial-time alternatives.


---
##  Section 6: Build Embedding Index with FAISS

We use `sentence-transformers` (`all-MiniLM-L6-v2`) to encode chunks into dense vectors, then build a **FAISS** flat L2 index for fast nearest-neighbour retrieval.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# ── Load embedding model ────────────────────────────────────────
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"   # Lightweight, 384-dim, strong retrieval quality
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print("✅ Embedding model loaded.")

# ── Encode all chunks ───────────────────────────────────────────
print(f"\nEncoding {len(all_chunks)} chunks...")
chunk_embeddings = embed_model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # Cosine similarity via inner product on unit vectors
)
print(f"✅ Embeddings shape: {chunk_embeddings.shape}")

# ── Build FAISS index ───────────────────────────────────────────
EMBEDDING_DIM = chunk_embeddings.shape[1]  # 384 for MiniLM

# IndexFlatIP: exact inner-product search (equivalent to cosine on normalised vectors)
faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
faiss_index.add(chunk_embeddings.astype(np.float32))

print(f"✅ FAISS index built.")
print(f"   Indexed vectors : {faiss_index.ntotal}")
print(f"   Embedding dim   : {EMBEDDING_DIM}")
print_vram_status("After indexing")

Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded.

Encoding 10 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings shape: (10, 384)
✅ FAISS index built.
   Indexed vectors : 10
   Embedding dim   : 384
[After indexing] VRAM — Allocated: 2.38 GB | Reserved: 2.44 GB | Free: 13.20 GB / 15.64 GB total


---
## 🔍 Section 7: Retrieval Function

In [ ]:
def retrieve(query: str, top_k: int = 3, min_score: float = 0.2) -> list[dict]:
    """
    Retrieve the top-k most relevant chunks for a given query.

    Args:
        query     : Natural-language question.
        top_k     : Number of chunks to retrieve.
        min_score : Minimum cosine-similarity score threshold.

    Returns:
        List of dicts with keys: rank, score, chunk, doc_id, doc_preview.
    """
    # Encode query with same normalisation
    query_vec = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    # Search FAISS index
    scores, indices = faiss_index.search(query_vec, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx == -1 or score < min_score:   # -1 = not found
            continue
        results.append({
            "rank"        : rank,
            "score"       : float(score),
            "chunk"       : all_chunks[idx],
            "doc_id"      : chunk_metadata[idx]["doc_id"],
            "doc_preview" : chunk_metadata[idx]["doc_preview"],
        })
    return results


# ── Quick retrieval test ────────────────────────────────────────
test_query = "What is the Erlang-C formula used for?"
test_results = retrieve(test_query, top_k=3)

print(f"Query: \"{test_query}\"\n")
for r in test_results:
    print(f"  Rank {r['rank']} | Score: {r['score']:.4f} | Doc: {r['doc_preview']}")
    print(f"  Chunk: {r['chunk'][:150]}...\n")

Query: "What is the Erlang-C formula used for?"

  Rank 1 | Score: 0.3658 | Doc: Queueing theory analyses waiting lines (queues) in
  Chunk: Queueing theory analyses waiting lines (queues) in systems such as call centres, hospitals, and ticket platforms. The M/M/1 model assumes Poisson arri...



---
## 🧩 Section 8: RAG Prompt Builder

In [ ]:
def build_rag_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    """
    Assemble a Llama-3 Instruct-formatted prompt that grounds the LLM
    response in the retrieved context.

    Format uses Llama-3's <|begin_of_text|> / <|im_start|> chat template.
    """
    # Build the context block from retrieved chunks
    context_lines = []
    for r in retrieved_chunks:
        context_lines.append(
            f"[Source {r['rank']} — relevance {r['score']:.2f}]\n{r['chunk']}"
        )
    context_block = "\n\n".join(context_lines)

    # Llama-3 chat template
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are a knowledgeable assistant specialising in Operations Research "
        "and Industrial Engineering. Answer the user's question strictly based "
        "on the provided context. If the context does not contain enough "
        "information, say so clearly. Be concise and technically precise."
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    return prompt


print("✅ Prompt builder defined.")
# Preview the prompt structure with the test query
sample_prompt = build_rag_prompt(test_query, test_results)
print(f"\nSample prompt length: {len(sample_prompt.split())} words")
print("\n--- Prompt preview (truncated) ---")
print(sample_prompt[:600], "...")

✅ Prompt builder defined.

Sample prompt length: 135 words

--- Prompt preview (truncated) ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a knowledgeable assistant specialising in Operations Research and Industrial Engineering. Answer the user's question strictly based on the provided context. If the context does not contain enough information, say so clearly. Be concise and technically precise.<|eot_id|><|start_header_id|>user<|end_header_id|>
Context:
[Source 1 — relevance 0.37]
Queueing theory analyses waiting lines (queues) in systems such as call centres, hospitals, and ticket platforms. The M/M/1 model assumes Poisson arrivals, exponential service times, a ...


---
## ⚙️ Section 9: Full RAG Generation Pipeline

In [ ]:
def rag_generate(
    query       : str,
    top_k       : int   = 3,
    max_new_tokens: int = 350,
    temperature : float = 0.3,
    top_p       : float = 0.9,
    verbose     : bool  = True
) -> dict:
    """
    End-to-end RAG pipeline:
        1. Retrieve top-k relevant chunks.
        2. Build grounded prompt.
        3. Generate response with the quantized LLM.

    Args:
        query          : User question.
        top_k          : Number of context chunks to retrieve.
        max_new_tokens : Maximum tokens to generate.
        temperature    : Sampling temperature (lower = more deterministic).
        top_p          : Nucleus sampling probability cutoff.
        verbose        : Print intermediate steps if True.

    Returns:
        dict with keys: query, retrieved_chunks, prompt, response, vram_gb.
    """
    if verbose:
        print(f"📥 Query: {query}")
        print("-" * 60)

    # ── Step 1: Retrieve ─────────────────────────────────────────
    retrieved = retrieve(query, top_k=top_k)

    if verbose:
        print(f"🔍 Retrieved {len(retrieved)} chunks:")
        for r in retrieved:
            print(f"   [{r['rank']}] Score={r['score']:.4f} | {r['doc_preview']}")
        print()

    # ── Step 2: Build prompt ─────────────────────────────────────
    prompt = build_rag_prompt(query, retrieved)

    # ── Step 3: Tokenise ─────────────────────────────────────────
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    ).to("cuda")

    if verbose:
        print(f"🔢 Prompt tokens: {inputs['input_ids'].shape[1]}")

    # ── Step 4: Generate ─────────────────────────────────────────
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens  = max_new_tokens,
            do_sample       = True,
            temperature     = temperature,
            top_p           = top_p,
            repetition_penalty = 1.1,      # Reduce repetition
            eos_token_id    = tokenizer.eos_token_id,
            pad_token_id    = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (strip the input)
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # ── VRAM snapshot ────────────────────────────────────────────
    vram_gb = torch.cuda.memory_allocated() / 1e9

    if verbose:
        print(f"\n🤖 Generated Response:")
        print("=" * 60)
        print(response)
        print("=" * 60)
        print(f"\n💾 VRAM in use: {vram_gb:.2f} GB")

    return {
        "query"            : query,
        "retrieved_chunks" : retrieved,
        "prompt"           : prompt,
        "response"         : response,
        "vram_gb"          : vram_gb,
    }

print("✅ RAG generation pipeline defined.")

✅ RAG generation pipeline defined.


---
## 🧪 Section 10: Live Query Demonstrations

In [ ]:
# ── Demo Query 1: Queueing Theory ───────────────────────────────
result1 = rag_generate(
    query="Explain the Erlang-C formula and what queueing metrics it provides.",
    top_k=3,
    max_new_tokens=300
)

📥 Query: Explain the Erlang-C formula and what queueing metrics it provides.
------------------------------------------------------------
🔍 Retrieved 3 chunks:
   [1] Score=0.7073 | Queueing theory analyses waiting lines (queues) in
   [2] Score=0.3537 | Inventory management balances holding costs agains
   [3] Score=0.3399 | Monte Carlo simulation uses random sampling to est

🔢 Prompt tokens: 429


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1


🤖 Generated Response:
The Erlang-C formula is a mathematical model used to calculate the probability that a customer must wait in an M/M/c queue, where:

* M represents a Markovian (Poisson) arrival process
* /c represents c parallel servers

The Erlang-C formula is given by:

P(calls remaining) = (λ/c)(1 - P0)

where:
- P(calls remaining) is the probability that there are no calls in the system
- λ is the arrival rate (average number of customers arriving per unit time)
- c is the number of servers
- P0 is the probability that all servers are busy (server utilization)

The Erlang-C formula provides several key queueing metrics:

1. **Probability of an idle system (P0)**: This is the probability that all servers are busy, which can be calculated using the Erlang-C formula.
2. **Average queue length (Lq)**: This metric measures the average number of customers waiting in the queue. It can be calculated using the formula Lq = ρ(1 - P0)/c, where ρ = λ/cμ is the server utilization factor.


In [ ]:
# ── Demo Query 2: Linear Programming ────────────────────────────
result2 = rag_generate(
    query="What is the simplex method and why is it used in linear programming?",
    top_k=3,
    max_new_tokens=300
)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📥 Query: What is the simplex method and why is it used in linear programming?
------------------------------------------------------------
🔍 Retrieved 3 chunks:
   [1] Score=0.7958 | Linear Programming (LP) is a mathematical optimisa
   [2] Score=0.4544 | Network optimisation models represent problems as 
   [3] Score=0.4178 | Integer Programming (IP) extends LP by requiring s

🔢 Prompt tokens: 430

🤖 Generated Response:
The simplex method is an algorithm for solving linear programming problems. It was developed by George Dantzig in 1947. The method involves iteratively improving the current basic feasible solution by moving to adjacent vertices of the feasible region until an optimal solution is reached. This is done by updating the basic variables and the right-hand side of the constraints, which allows the algorithm to converge to the optimal solution.

The simplex method is used in linear programming because it provides a systematic way to explore the entire feasible region and ide

In [ ]:
# ── Demo Query 3: Inventory Management ──────────────────────────
result3 = rag_generate(
    query="How does the Economic Order Quantity model minimise inventory costs?",
    top_k=3,
    max_new_tokens=300
)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📥 Query: How does the Economic Order Quantity model minimise inventory costs?
------------------------------------------------------------
🔍 Retrieved 3 chunks:
   [1] Score=0.6996 | Inventory management balances holding costs agains
   [2] Score=0.4008 | Dynamic Programming (DP) solves complex problems b
   [3] Score=0.3428 | Network optimisation models represent problems as 

🔢 Prompt tokens: 410

🤖 Generated Response:
The Economic Order Quantity (EOQ) model minimizes inventory costs by optimizing the trade-off between holding costs and ordering (setup) costs. Specifically, it aims to find the optimal quantity of inventory to order at each time period such that the total inventory cost is minimized.

In the EOQ model, the total inventory cost consists of two components:

1. Holding cost: This is the cost of maintaining inventory over time.
2. Ordering cost: This is the cost of placing an order to restock the inventory.

The EOQ model assumes a deterministic, constant demand rate, whi

In [ ]:
# ── Demo Query 4: Out-of-domain (graceful degradation) ──────────
result4 = rag_generate(
    query="What are the rules of cricket?",
    top_k=3,
    max_new_tokens=150
)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📥 Query: What are the rules of cricket?
------------------------------------------------------------
🔍 Retrieved 0 chunks:

🔢 Prompt tokens: 72

🤖 Generated Response:
A complex and nuanced sport!

The rules of cricket can be summarized as follows:

**Objective:** The objective of cricket is to score runs by hitting the ball with a bat and running between two sets of three stumps (wickets) while the opposing team tries to stop them.

**Gameplay:**

1. **Match format:** A standard match consists of two innings per team.
2. **Innings:** An innings is divided into overs, which consist of six balls bowled by the opposing team.
3. **Overs:** Each over begins with a bowler delivering the ball from one end of the pitch, followed by a batsman at the opposite end.
4. **Batting:** A batsman attempts to hit the ball with

💾 VRAM in use: 2.49 GB


---
## 🖊️ Section 11: Interactive Query Interface

In [ ]:
# ── Ask your own question ────────────────────────────────────────
# Change USER_QUERY to any question related to your document corpus.

USER_QUERY = "Explain the difference between weighted and lexicographic goal programming."

custom_result = rag_generate(
    query          = USER_QUERY,
    top_k          = 4,
    max_new_tokens = 400,
    temperature    = 0.3,
)

Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📥 Query: Explain the difference between weighted and lexicographic goal programming.
------------------------------------------------------------
🔍 Retrieved 4 chunks:
   [1] Score=0.6355 | Goal Programming (GP) is a multi-objective optimis
   [2] Score=0.4120 | Linear Programming (LP) is a mathematical optimisa
   [3] Score=0.3467 | Dynamic Programming (DP) solves complex problems b
   [4] Score=0.3092 | Integer Programming (IP) extends LP by requiring s

🔢 Prompt tokens: 521

🤖 Generated Response:
Weighted Goal Programming (WGP) and Lexicographic Goal Programming (LGP) are both multi-objective optimization techniques used to minimize deviations from target levels for multiple objectives. The key difference between the two lies in how they prioritize and optimize these objectives.

**Weighted Goal Programming (WGP):**

In WGP, the objective function is a weighted sum of deviations from target levels. Each deviation variable (d+) and (d-) captures the gap between achieved and desired l

---
## 📊 Section 12: VRAM Usage Summary & Benchmarking

In [ ]:
import pandas as pd

# ── VRAM summary ─────────────────────────────────────────────────
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
alloc_vram = torch.cuda.memory_allocated() / 1e9
resrv_vram = torch.cuda.memory_reserved()  / 1e9
free_vram  = total_vram - resrv_vram

print("=" * 55)
print("       VRAM USAGE SUMMARY (Dynamic 4-bit RAG)")
print("=" * 55)
print(f"  GPU Model      : {torch.cuda.get_device_name(0)}")
print(f"  Total VRAM     : {total_vram:.2f} GB")
print(f"  Allocated      : {alloc_vram:.2f} GB  ({alloc_vram/total_vram*100:.1f}%)")
print(f"  Reserved       : {resrv_vram:.2f} GB  ({resrv_vram/total_vram*100:.1f}%)")
print(f"  Free           : {free_vram:.2f} GB  ({free_vram/total_vram*100:.1f}%)")
print("=" * 55)

# ── Per-query VRAM table ─────────────────────────────────────────
queries_summary = [
    {"Query": result1["query"][:55] + "...", "VRAM (GB)": f"{result1['vram_gb']:.2f}"},
    {"Query": result2["query"][:55] + "...", "VRAM (GB)": f"{result2['vram_gb']:.2f}"},
    {"Query": result3["query"][:55] + "...", "VRAM (GB)": f"{result3['vram_gb']:.2f}"},
    {"Query": result4["query"][:55] + "...", "VRAM (GB)": f"{result4['vram_gb']:.2f}"},
]
df_vram = pd.DataFrame(queries_summary)
print("\nPer-query VRAM snapshot:")
print(df_vram.to_string(index=False))

# ── Quantization efficiency note ────────────────────────────────
print("\n" + "─" * 55)
print("Dynamic 4-bit quantization benefit:")
print("  • Standard FP16 Llama-3.2-3B  ≈ 6.4 GB VRAM")
print("  • Static 4-bit                ≈ 2.0 GB VRAM")
print("  • Unsloth dynamic 4-bit       ≈ 2.0–2.5 GB VRAM")
print("    (critical layers kept at higher precision)")
print("─" * 55)

       VRAM USAGE SUMMARY (Dynamic 4-bit RAG)
  GPU Model      : Tesla T4
  Total VRAM     : 15.64 GB
  Allocated      : 2.54 GB  (16.2%)
  Reserved       : 2.71 GB  (17.3%)
  Free           : 12.93 GB  (82.7%)

Per-query VRAM snapshot:
                                                     Query VRAM (GB)
Explain the Erlang-C formula and what queueing metrics ...      2.53
What is the simplex method and why is it used in linear...      2.53
How does the Economic Order Quantity model minimise inv...      2.53
                         What are the rules of cricket?...      2.49

───────────────────────────────────────────────────────
Dynamic 4-bit quantization benefit:
  • Standard FP16 Llama-3.2-3B  ≈ 6.4 GB VRAM
  • Static 4-bit                ≈ 2.0 GB VRAM
  • Unsloth dynamic 4-bit       ≈ 2.0–2.5 GB VRAM
    (critical layers kept at higher precision)
───────────────────────────────────────────────────────


---
## 🔄 Section 13: (Optional) Swap to a Larger Model

Uncomment and run this cell to upgrade to a more capable 7B model — requires ~5 GB VRAM on T4.

In [ ]:
# # ── Upgrade to Mistral-7B for higher quality responses ──────────
# # Requires T4 (16 GB) with dynamic 4-bit to fit within VRAM budget.

# del model, tokenizer            # Free current model
# torch.cuda.empty_cache()
# print_vram_status("After freeing Llama-3.2-3B")

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name     = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
#     max_seq_length = MAX_SEQ_LENGTH,
#     dtype          = None,
#     load_in_4bit   = True,
# )
# FastLanguageModel.for_inference(model)
# print_vram_status("After loading Mistral-7B dynamic 4-bit")

# # Re-run any rag_generate() call with the same API
# result_mistral = rag_generate(
#     query="Explain Markov Decision Processes and their use in inventory control.",
#     top_k=4,
#     max_new_tokens=400
# )

---
## ✅ Summary

| Component | Choice | Why |
|---|---|---|
| **LLM** | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` | Dynamic 4-bit: quality near FP16 at ~2 GB VRAM |
| **Quantization** | Unsloth dynamic `bnb-4bit` | Selectively preserves precision for critical attention weights |
| **Embeddings** | `all-MiniLM-L6-v2` (384-dim) | Fast, accurate, runs on CPU |
| **Vector Store** | FAISS `IndexFlatIP` | Exact cosine search, no approximation error |
| **Chunking** | 150-word windows, 30-word overlap | Balances context richness and retrieval precision |
| **Prompt style** | Llama-3 Instruct chat template | Matches model fine-tuning format |
| **VRAM** | ~2–3 GB active | Fits T4 with large headroom for long contexts |

**Pipeline flow:**
```
User Query
    │
    ▼
[Sentence-Transformer Encoder]
    │  384-dim query vector
    ▼
[FAISS Index]  ──→  Top-k chunks (cosine similarity)
    │
    ▼
[Prompt Builder]  ──→  System + Context + Question
    │
    ▼
[Unsloth Dynamic 4-bit LLM]  ──→  Grounded Response
```